# Simple Token Method
**Written:** <u>June 16 - 19, 2026</u><br>
**Published:** <u>June 19, 2026</u>

- This method focuses on retaining "important" sentences while removing the "non-important" sentences through a scoring method.


> Dataset source: [AsadMahmood (Kaggle)](https://www.kaggle.com/datasets/asad1m9a9h6mood/news-articles)

In [1]:
# Libraries
import pandas as pd
import re
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from collections import Counter
from heapq import nlargest

In [2]:
# Load Dataset
df = pd.read_csv("Articles.csv", encoding="cp1252")  # file encoding difference (cp1252 works)

print(df.head())
print(df.shape)

                                             Article      Date  \
0  KARACHI: The Sindh government has decided to b...  1/1/2015   
1  HONG KONG: Asian markets started 2015 on an up...  1/2/2015   
2  HONG KONG:  Hong Kong shares opened 0.66 perce...  1/5/2015   
3  HONG KONG: Asian markets tumbled Tuesday follo...  1/6/2015   
4  NEW YORK: US oil prices Monday slipped below $...  1/6/2015   

                                             Heading  NewsType  
0  sindh govt decides to cut public transport far...  business  
1                    asia stocks up in new year trad  business  
2           hong kong stocks open 0.66 percent lower  business  
3             asian stocks sink euro near nine year   business  
4                 us oil prices slip below 50 a barr  business  
(2692, 4)


In [3]:
# Tokenize sample
article = df["Article"][0].strip()  # can select from index 0 to 2691
article = re.sub(r'\.(?=[A-Z])', '. ', article)  # fastest way to add spaces between sentences

sentences = sent_tokenize(article)
words = word_tokenize(article.lower())

In [4]:
# Tokenize sample sentences
print(sentences)

['KARACHI: The Sindh government has decided to bring down public transport fares by 7 per cent due to massive reduction in petroleum product prices by the federal government, Geo News reported.', 'Sources said reduction in fares will be applicable on public transport, rickshaw, taxi and other means of traveling.', 'Meanwhile, Karachi Transport Ittehad (KTI) has refused to abide by the government decision.', 'KTI President Irshad Bukhari said the commuters are charged the lowest fares in Karachi as compare to other parts of the country, adding that 80pc vehicles run on Compressed Natural Gas (CNG).', 'Bukhari said Karachi transporters will cut fares when decrease in CNG prices will be made.']


In [5]:
# Tokenize sample words
print(words)

['karachi', ':', 'the', 'sindh', 'government', 'has', 'decided', 'to', 'bring', 'down', 'public', 'transport', 'fares', 'by', '7', 'per', 'cent', 'due', 'to', 'massive', 'reduction', 'in', 'petroleum', 'product', 'prices', 'by', 'the', 'federal', 'government', ',', 'geo', 'news', 'reported', '.', 'sources', 'said', 'reduction', 'in', 'fares', 'will', 'be', 'applicable', 'on', 'public', 'transport', ',', 'rickshaw', ',', 'taxi', 'and', 'other', 'means', 'of', 'traveling', '.', 'meanwhile', ',', 'karachi', 'transport', 'ittehad', '(', 'kti', ')', 'has', 'refused', 'to', 'abide', 'by', 'the', 'government', 'decision', '.', 'kti', 'president', 'irshad', 'bukhari', 'said', 'the', 'commuters', 'are', 'charged', 'the', 'lowest', 'fares', 'in', 'karachi', 'as', 'compare', 'to', 'other', 'parts', 'of', 'the', 'country', ',', 'adding', 'that', '80pc', 'vehicles', 'run', 'on', 'compressed', 'natural', 'gas', '(', 'cng', ')', '.', 'bukhari', 'said', 'karachi', 'transporters', 'will', 'cut', 'fares

In [6]:
# Remove stopwords
stop_words = set(stopwords.words("english"))

filtered = []

for word in words:
    if word.isalnum() and word not in stop_words:
        filtered.append(word)

In [7]:
# Check
print(filtered)

['karachi', 'sindh', 'government', 'decided', 'bring', 'public', 'transport', 'fares', '7', 'per', 'cent', 'due', 'massive', 'reduction', 'petroleum', 'product', 'prices', 'federal', 'government', 'geo', 'news', 'reported', 'sources', 'said', 'reduction', 'fares', 'applicable', 'public', 'transport', 'rickshaw', 'taxi', 'means', 'traveling', 'meanwhile', 'karachi', 'transport', 'ittehad', 'kti', 'refused', 'abide', 'government', 'decision', 'kti', 'president', 'irshad', 'bukhari', 'said', 'commuters', 'charged', 'lowest', 'fares', 'karachi', 'compare', 'parts', 'country', 'adding', '80pc', 'vehicles', 'run', 'compressed', 'natural', 'gas', 'cng', 'bukhari', 'said', 'karachi', 'transporters', 'cut', 'fares', 'decrease', 'cng', 'prices', 'made']


In [8]:
# Count word frequency
freq = Counter(filtered)

In [9]:
# Check word frequency
print(freq)

Counter({'karachi': 4, 'fares': 4, 'government': 3, 'transport': 3, 'said': 3, 'public': 2, 'reduction': 2, 'prices': 2, 'kti': 2, 'bukhari': 2, 'cng': 2, 'sindh': 1, 'decided': 1, 'bring': 1, '7': 1, 'per': 1, 'cent': 1, 'due': 1, 'massive': 1, 'petroleum': 1, 'product': 1, 'federal': 1, 'geo': 1, 'news': 1, 'reported': 1, 'sources': 1, 'applicable': 1, 'rickshaw': 1, 'taxi': 1, 'means': 1, 'traveling': 1, 'meanwhile': 1, 'ittehad': 1, 'refused': 1, 'abide': 1, 'decision': 1, 'president': 1, 'irshad': 1, 'commuters': 1, 'charged': 1, 'lowest': 1, 'compare': 1, 'parts': 1, 'country': 1, 'adding': 1, '80pc': 1, 'vehicles': 1, 'run': 1, 'compressed': 1, 'natural': 1, 'gas': 1, 'transporters': 1, 'cut': 1, 'decrease': 1, 'made': 1})


In [10]:
# Score each sentence
scores = {}

for sentence in sentences:  # loop through each sentence
    for word in word_tokenize(sentence.lower()):  # tokenize per sentence
        if word in freq:
            scores[sentence] = scores.get(sentence, 0) + freq[word]

In [11]:
print(scores)

{'KARACHI: The Sindh government has decided to bring down public transport fares by 7 per cent due to massive reduction in petroleum product prices by the federal government, Geo News reported.': 37, 'Sources said reduction in fares will be applicable on public transport, rickshaw, taxi and other means of traveling.': 20, 'Meanwhile, Karachi Transport Ittehad (KTI) has refused to abide by the government decision.': 17, 'KTI President Irshad Bukhari said the commuters are charged the lowest fares in Karachi as compare to other parts of the country, adding that 80pc vehicles run on Compressed Natural Gas (CNG).': 32, 'Bukhari said Karachi transporters will cut fares when decrease in CNG prices will be made.': 21}


In [12]:
summary = nlargest(3, scores, key=scores.get)  # filter top 3 sentences

print("ORIGINAL:")
print(article)
print()
print("SUMMARY:")
print(" ".join(summary))

ORIGINAL:
KARACHI: The Sindh government has decided to bring down public transport fares by 7 per cent due to massive reduction in petroleum product prices by the federal government, Geo News reported. Sources said reduction in fares will be applicable on public transport, rickshaw, taxi and other means of traveling. Meanwhile, Karachi Transport Ittehad (KTI) has refused to abide by the government decision. KTI President Irshad Bukhari said the commuters are charged the lowest fares in Karachi as compare to other parts of the country, adding that 80pc vehicles run on Compressed Natural Gas (CNG). Bukhari said Karachi transporters will cut fares when decrease in CNG prices will be made.

SUMMARY:
KARACHI: The Sindh government has decided to bring down public transport fares by 7 per cent due to massive reduction in petroleum product prices by the federal government, Geo News reported. KTI President Irshad Bukhari said the commuters are charged the lowest fares in Karachi as compare to o

# More Advanced
Using the simple method (word frequency per sentence), TF-IDF (Term Frequency-Inverse Document Frequency), text rank we create a text summarizer with a simple UI.

NOTE THIS CODE WAS ASSISTED BY CHATGPT.

> Dataset source: [AsadMahmood (Kaggle)](https://www.kaggle.com/datasets/asad1m9a9h6mood/news-articles)

In [13]:
# Libraries
import pandas as pd
import numpy as np
import networkx as nx
import re
import nltk

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
from collections import Counter
from heapq import nlargest

In [14]:
# Stop words
stop_words = set(stopwords.words("english"))

In [15]:
# Data preprocessing class
class Preprocessor:

    @staticmethod
    def sentences(text):
        text = re.sub(r'\.(?=[A-Z])', '. ', text)  # spaces between sentences
        text = re.sub(r'\s+', ' ', text).strip()  # cleans spacing
        return sent_tokenize(text)
    
    @staticmethod
    def clean(text):
        text = text.lower()
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # only take alphanumeric characters
        return text.strip()

    @staticmethod
    def remove_stopwords(words):
        return [word for word in words if word not in stop_words]

In [16]:
# Base class
class BaseSummarizer:

    def preprocess(self, article):
        original = Preprocessor.sentences(article)  # get sentence tokens
        cleaned = [Preprocessor.clean(s) for s in original]  # clean
        return original, cleaned

    def summarize(self, article):
        raise NotImplementedError  # error catcher (no method found)

In [17]:
# Simple method (word frequency per sentence)
class FrequencySummarizer(BaseSummarizer):  # similar to the imlpementation above (added a feature that retains original text with punctuations)

    def summarize(self, article):
        original, cleaned = self.preprocess(article)
        words = word_tokenize(Preprocessor.clean(article))
        words = Preprocessor.remove_stopwords(words)
        frequency = Counter(words)

        scores = {}
        
        for original_s, cleaned_s in zip(original, cleaned):
            for word in word_tokenize(cleaned_s):
                if word in frequency:
                    scores[original_s] = scores.get(original_s, 0) + frequency[word]
    
        summary = nlargest(3, scores, key=scores.get)
        return " ".join(summary)

In [18]:
# TF-IDF (term frequency - inverse document frequency)
class TFIDFSummarizer(BaseSummarizer):  # how frequent is a word in a sentence - how rare or common is that word across the entire article

    def summarize(self, article):
        original, cleaned = self.preprocess(article)
        matrix = TfidfVectorizer().fit_transform(cleaned)  # matrix = rows:sentences and columns:words then get TF-IDF
        
        scores = np.asarray(matrix.sum(axis=1)).flatten()  # add up all the scores
        best = nlargest(3, range(len(scores)), scores.take)  # extract the top 3
    
        summary = [original[i] for i in best]
        return " ".join(summary)

In [19]:
# Text rank
class TextRankSummarizer(BaseSummarizer):  # same as TF-IDF with an addition of cosine similarity

    def summarize(self, article):
        original, cleaned = self.preprocess(article)
        matrix = TfidfVectorizer().fit_transform(cleaned)
        similarity = cosine_similarity(matrix)  # measures the angle between vectors (TF-IDF) in a geometric space
        graph = nx.from_numpy_array(similarity)  # converts that table into a living network graph
    
        scores = nx.pagerank(graph)
        ranked = sorted(scores, key=scores.get, reverse=True)
        
        summary = [original[i] for i in ranked[:3]]
        return " ".join(summary)

In [20]:
# Evaluation
class Evaluator:  # uses ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

    @staticmethod
    def rouge(reference, prediction):
        scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        return scorer.score(reference, prediction)

***A little more explanation on ROUGE***  
In the simplest words, rouge1 (word count - did the AI mention the right topics and keywords), rouge2 (phrasing - did the AI capture the right descriptions and context), rougeL (flow and structure - did the AI keep the correct word order and sentence structure). Higher is better for all three.

In [24]:
# Basic UI
CSV_PATH = "Articles.csv"
df = pd.read_csv(CSV_PATH, encoding="cp1252")
df["Article"] = df["Article"].str.replace(r'\.(?=[A-Z])', '. ', regex=True)
df["Article"] = df["Article"].str.replace(r'\s+', ' ', regex=True).str.strip()

frequency = FrequencySummarizer()
tfidf = TFIDFSummarizer()
textrank = TextRankSummarizer()
evaluator = Evaluator()  # Instantiate the Evaluator

while True:
    print("=" * 60)
    print("TEXT SUMMARIZATION COMPARISON")
    print("=" * 60)

    index = input("\nEnter article index (0-2691) or q to quit: ")

    if index.lower() == "q":
        break

    try:
        index = int(index)
        if index < 0 or index >= len(df):
            print("\nInvalid index!\n")
            continue
    except ValueError:
        print("\nPlease enter a valid number.\n")
        continue

    heading = df.loc[index, "Heading"]
    article = df.loc[index, "Article"]

    # Generate Summaries
    frequency_summary = frequency.summarize(article)
    tfidf_summary = tfidf.summarize(article)
    textrank_summary = textrank.summarize(article)

    # Calculate ROUGE Scores (comparing each summary back to the original full article)
    freq_scores = evaluator.rouge(article, frequency_summary)
    tfidf_scores = evaluator.rouge(article, tfidf_summary)
    textrank_scores = evaluator.rouge(article, textrank_summary)

    print("\n")
    print("=" * 60)
    print("ORIGINAL HEADING")
    print("=" * 60)
    print(heading)

    print("\n")
    print("=" * 60)
    print("ORIGINAL ARTICLE")
    print("=" * 60)
    print(article)

    print("\n")
    print("=" * 60)
    print("LEVEL 1 - FREQUENCY")
    print("=" * 60)
    print(frequency_summary)
    print("-" * 60)
    print(f"ROUGE-1 F1: {freq_scores['rouge1'].fmeasure:.4f} | "
          f"ROUGE-2 F1: {freq_scores['rouge2'].fmeasure:.4f} | "
          f"ROUGE-L F1: {freq_scores['rougeL'].fmeasure:.4f}")

    print("\n")
    print("=" * 60)
    print("LEVEL 2 - TF-IDF")
    print("=" * 60)
    print(tfidf_summary)
    print("-" * 60)
    print(f"ROUGE-1 F1: {tfidf_scores['rouge1'].fmeasure:.4f} | "
          f"ROUGE-2 F1: {tfidf_scores['rouge2'].fmeasure:.4f} | "
          f"ROUGE-L F1: {tfidf_scores['rougeL'].fmeasure:.4f}")

    print("\n")
    print("=" * 60)
    print("LEVEL 3 - TEXTRANK")
    print("=" * 60)
    print(textrank_summary)
    print("-" * 60)
    print(f"ROUGE-1 F1: {textrank_scores['rouge1'].fmeasure:.4f} | "
          f"ROUGE-2 F1: {textrank_scores['rouge2'].fmeasure:.4f} | "
          f"ROUGE-L F1: {textrank_scores['rougeL'].fmeasure:.4f}")
    print()

TEXT SUMMARIZATION COMPARISON



Enter article index (0-2691) or q to quit:  0




ORIGINAL HEADING
sindh govt decides to cut public transport fares by 7pc kti rej


ORIGINAL ARTICLE
KARACHI: The Sindh government has decided to bring down public transport fares by 7 per cent due to massive reduction in petroleum product prices by the federal government, Geo News reported. Sources said reduction in fares will be applicable on public transport, rickshaw, taxi and other means of traveling. Meanwhile, Karachi Transport Ittehad (KTI) has refused to abide by the government decision. KTI President Irshad Bukhari said the commuters are charged the lowest fares in Karachi as compare to other parts of the country, adding that 80pc vehicles run on Compressed Natural Gas (CNG). Bukhari said Karachi transporters will cut fares when decrease in CNG prices will be made.


LEVEL 1 - FREQUENCY
KARACHI: The Sindh government has decided to bring down public transport fares by 7 per cent due to massive reduction in petroleum product prices by the federal government, Geo News reported.


Enter article index (0-2691) or q to quit:  q


<u>Future Plans</u>  
explore the deeper meaninigs per concept. For now we explore what it does, do, tells us.